In [1]:
import os
import sys
import logging
import random
import numpy as np

# Keep Kaggle output readable while TensorFlow initializes.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)
logging.getLogger("tensorflow").setLevel(logging.FATAL)

import tensorflow as tf

# Fixed seeds make the comparison as reproducible as practical.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
# REPOSITORY CONFIGURATION

REPO_NAME = "RefraScan"
GITHUB_USER = "KyziaPi"
BRANCH_NAME = "Model-Experiment"   # <-- point this at your branch

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
REPO_PATH = os.path.join("/kaggle/working", REPO_NAME)

if not os.path.exists(REPO_PATH):
    print(f"Cloning {REPO_NAME} branch '{BRANCH_NAME}'...")
    !env GIT_TERMINAL_PROMPT=0 git clone -b {BRANCH_NAME} {REPO_URL}
else:
    print(f"{REPO_NAME} already exists. Updating branch '{BRANCH_NAME}'...")
    !cd {REPO_PATH} && env GIT_TERMINAL_PROMPT=0 git fetch --all && git checkout {BRANCH_NAME} && git pull origin {BRANCH_NAME}

if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

import sys
import math
import pandas as pd
from tensorflow.keras.applications.resnet50 import preprocess_input

from src.preprocessing import (
    load_and_clean_data,
    encode_target,
    validate_dataset,
    majority_class_baseline,
)
from src.cross_validation import run_cross_validation, split_holdout_test
from src.evaluate import majority_class_baseline_metrics

print(f"Environment configured successfully! Working on branch: {BRANCH_NAME}")


Cloning RefraScan branch 'Model-Experiment'...
Cloning into 'RefraScan'...
remote: Enumerating objects: 475, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 475 (delta 87), reused 79 (delta 36), pack-reused 344 (from 1)
Receiving objects: 100% (475/475), 48.67 MiB | 34.44 MiB/s, done.
Resolving deltas: 100% (261/261), done.
Environment configured successfully! Working on branch: Model-Experiment


In [3]:
# DATASET VALIDATION + MAJORITY-CLASS BASELINE

DATASET_DIR = '/kaggle/input/datasets/yerikaelainegueco/fundus-images-with-refractive-values'
CSV_PATH = os.path.join(DATASET_DIR, 'RefraScan_dataset.csv')
IMG_DIR = os.path.join(DATASET_DIR, 'FundusImages')

# Load the original records. No refractive measurement is passed to the model.
df = load_and_clean_data(CSV_PATH, IMG_DIR)

# Validate images, labels, patient grouping, and target-derived columns BEFORE training.
df = validate_dataset(df, patient_col="ID", target_col="classification")

# Encode only the three clinical target classes.
df = encode_target(df)

# Establish the descriptive majority-class baseline on the development pool.
development_df, holdout_df = split_holdout_test(
    df,
    patient_col="ID",
    target_col="classification_encoded",
    test_size=0.15,
)

baseline = majority_class_baseline_metrics(
    development_df["classification_encoded"].values
)

print("\n--- Majority-Class Baseline (Development Data) ---")
print(f"Majority class : {baseline['majority_class_name']}")
print(f"Accuracy       : {baseline['accuracy']:.4f}")
print(f"Balanced Acc.  : {baseline['balanced_accuracy']:.4f}")
print(f"Macro F1       : {baseline['macro_f1']:.4f}")
print("\nDevelopment class distribution:")
print(development_df["classification"].value_counts())
print("\nHoldout class distribution (kept untouched):")
print(holdout_df["classification"].value_counts())



DATASET VALIDATION

Total records       : 1,018
Unique patients     : 517
Missing patient IDs : 0
Missing labels      : 0
Duplicate rows      : 0

Class distribution:
classification
Myopia        660
Hyperopia     236
Emmetropia    122
Name: count, dtype: int64

Patients with multiple target classes: 50
These patients have different classifications between their eyes. This is allowed.

Example mixed-class patients:
 ID classification
  2     Emmetropia
  2         Myopia
  4     Emmetropia
  4      Hyperopia
  7     Emmetropia
  7         Myopia
 14         Myopia
 14     Emmetropia
 19     Emmetropia
 19      Hyperopia
 24      Hyperopia
 24     Emmetropia
 27     Emmetropia
 27         Myopia
 34     Emmetropia
 34      Hyperopia
 42     Emmetropia
 42         Myopia
 44         Myopia
 44      Hyperopia

Valid target classes confirmed:
['Emmetropia', 'Hyperopia', 'Myopia']

Target-derived refractive measurement columns detected:
  - sphere
  - cylinder
  - spherical_equivalent

The

In [4]:
# CONTROLLED IMAGE-ONLY 10-FOLD COMPARISON

# The three architecture notebooks use the same: 
#   * 15% patient-level holdout rule
#   * 85% development data
#   * 10-fold Stratified Group CV
#   * 224x224 input size
#   * image preprocessing and mild augmentation
#   * focal loss + training-fold class weights
#   * primary metrics: Macro F1 and balanced accuracy
#   * secondary metric: accuracy

# IMPORTANT: use_metadata=False means sphere, cylinder, spherical equivalent,
# and age are NOT inputs in this architecture-comparison stage.
results = run_cross_validation(
    df=df,
    model_name="resnet50",
    preprocess_input=preprocess_input,
    patient_col="ID",
    target_col="classification_encoded",
    n_splits=10,
    batch_size=16,
    epochs=30,
    learning_rate=1e-4,
    use_metadata=False,
    output_dir=f"/kaggle/working/{REPO_NAME}/artifacts",
    fine_tune=False,
)

# Save fold metrics so the three notebooks can be compared consistently.
results["fold_results"].to_csv(
    f"/kaggle/working/{REPO_NAME}/resnet50_image_only_cv_results.csv",
    index=False,
)

print("\nImage-only experiment completed for ResNet50.")



PATIENT-LEVEL HOLDOUT SPLIT
Development records : 864
Holdout records     : 154
Development patients: 439
Holdout patients    : 78
Patient overlap     : 0

The holdout set is now untouched and will not be used for architecture/model selection.

RESNET50 | IMAGE ONLY
10-FOLD STRATIFIED GROUP CROSS-VALIDATION

--- Fold 1/10 ---


I0000 00:00:1786709340.594972      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786709340.597798      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/30
 2/49 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.2969 - loss: 0.7827  

I0000 00:00:1786709359.511164     145 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 610ms/step - accuracy: 0.4708 - loss: 0.8263
Epoch 1: val_loss improved from None to 0.43112, saving model to /kaggle/working/RefraScan/artifacts/resnet50_image_fold_1_frozen.weights.h5

Epoch 1: finished saving model to /kaggle/working/RefraScan/artifacts/resnet50_image_fold_1_frozen.weights.h5
49/49 ━━━━━━━━━━━━━━━━━━━━ 55s 844ms/step - accuracy: 0.5571 - loss: 0.6750 - val_accuracy: 0.7294 - val_loss: 0.4311
Epoch 2/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step - accuracy: 0.6601 - loss: 0.4029
Epoch 2: val_loss did not improve from 0.43112
49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 331ms/step - accuracy: 0.6418 - loss: 0.4578 - val_accuracy: 0.5625 - val_loss: 0.5610
Epoch 3/30
 1/49 ━━━━━━━━━━━━━━━━━━━━ 6s 134ms/step - accuracy: 0.6875 - loss: 0.2863

...

49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 390ms/step - accuracy: 0.6903 - loss: 0.3614
Epoch 3: val_loss did not improve from 0.45861
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 395ms/step - accuracy: 0.6559 - loss: 0.4174 - val_accuracy: 0.5625 - val_loss: 0.6397
Epoch 4/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 379ms/step - accuracy: 0.6297 - loss: 0.4808
Epoch 4: val_loss did not improve from 0.45861
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 383ms/step - accuracy: 0.6559 - loss: 0.4275 - val_accuracy: 0.5312 - val_loss: 0.5970
Epoch 5/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step - accuracy: 0.6602 - loss: 0.3932
Epoch 5: val_loss did not improve from 0.45861
49/49 ━━━━━━━━━━━━━━━━━━━━ 17s 348ms/step - accuracy: 0.6727 - loss: 0.3867 - val_accuracy: 0.5000 - val_loss: 0.6793
Epoch 6/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 364ms/step - accuracy: 0.6964 - loss: 0.3671
Epoch 6: val_loss did not improve from 0.45861
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 368ms/step - accuracy: 0.6753 - loss: 0.3684 - val_accuracy: 0.5000 - val_loss: 0.7252
Epoch 7

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step - accuracy: 0.6627 - loss: 0.3933
Epoch 3: val_loss did not improve from 0.38303
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 386ms/step - accuracy: 0.6462 - loss: 0.4103 - val_accuracy: 0.1875 - val_loss: 0.6002
Epoch 4/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step - accuracy: 0.6442 - loss: 0.3991
Epoch 4: val_loss did not improve from 0.38303
49/49 ━━━━━━━━━━━━━━━━━━━━ 17s 341ms/step - accuracy: 0.6769 - loss: 0.3810 - val_accuracy: 0.2500 - val_loss: 0.6221
Epoch 5/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step - accuracy: 0.6376 - loss: 0.3866
Epoch 5: val_loss did not improve from 0.38303
49/49 ━━━━━━━━━━━━━━━━━━━━ 17s 348ms/step - accuracy: 0.6615 - loss: 0.3660 - val_accuracy: 0.2500 - val_loss: 0.5983
Epoch 6/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.7043 - loss: 0.3510
Epoch 6: val_loss did not improve from 0.38303
49/49 ━━━━━━━━━━━━━━━━━━━━ 17s 344ms/step - accuracy: 0.6897 - loss: 0.3461 - val_accuracy: 0.2500 - val_loss: 0.6040
Epoch 7

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 385ms/step - accuracy: 0.6159 - loss: 0.4394
Epoch 3: val_loss did not improve from 0.41710
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 389ms/step - accuracy: 0.6374 - loss: 0.4328 - val_accuracy: 0.2500 - val_loss: 0.8053
Epoch 4/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 392ms/step - accuracy: 0.6829 - loss: 0.4133
Epoch 4: val_loss did not improve from 0.41710
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 396ms/step - accuracy: 0.6852 - loss: 0.3970 - val_accuracy: 0.2812 - val_loss: 0.6585
Epoch 5/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 395ms/step - accuracy: 0.6805 - loss: 0.3972
Epoch 5: val_loss did not improve from 0.41710
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 399ms/step - accuracy: 0.6774 - loss: 0.3632 - val_accuracy: 0.2812 - val_loss: 0.7164
Epoch 6/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 389ms/step - accuracy: 0.6828 - loss: 0.4006
Epoch 6: val_loss did not improve from 0.41710
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 393ms/step - accuracy: 0.6852 - loss: 0.3665 - val_accuracy: 0.3125 - val_loss: 0.6937
Epoch 7